# Bronze Layer - CDC Event Ingestion

This notebook ingests raw CDC events from Kafka into Delta tables with **schema drift detection** and monitoring.

In [ ]:
# Run helper notebooks
%run ./helpers/NB_catalog_helpers
%run ./helpers/NB_schema_drift_helpers
%run ./helpers/NB_schema_contracts

In [ ]:
from pyspark.sql.functions import col, current_timestamp, regexp_extract, lit, to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, TimestampType
import uuid

dbutils.widgets.text("KAFKA_BOOTSTRAP", "")
dbutils.widgets.text("TOPIC_PATTERN", "cdc.public.*")
dbutils.widgets.text("CATALOG", "workspace")
dbutils.widgets.text("BRONZE_SCHEMA", "bronze")
dbutils.widgets.text("CHECKPOINT_PATH", "/Volumes/workspace/default/mnt/checkpoints/bronze_cdc")
dbutils.widgets.text("MONITORING_SCHEMA", "monitoring")
dbutils.widgets.text("SCHEMA_POLICY", "additive_only")  # strict, additive_only, permissive
dbutils.widgets.text("ALERT_CHANNEL", "log")  # log, webhook, all
dbutils.widgets.text("WEBHOOK_URL", "")  # Optional Slack/Teams webhook

CONFIG = {
    "kafka_bootstrap": dbutils.widgets.get("KAFKA_BOOTSTRAP"),
    "topic_pattern": dbutils.widgets.get("TOPIC_PATTERN"),
    "catalog": dbutils.widgets.get("CATALOG"),
    "bronze_schema": dbutils.widgets.get("BRONZE_SCHEMA"),
    "checkpoint_path": dbutils.widgets.get("CHECKPOINT_PATH"),
    "monitoring_schema": dbutils.widgets.get("MONITORING_SCHEMA"),
    "schema_policy": dbutils.widgets.get("SCHEMA_POLICY"),
    "alert_channel": dbutils.widgets.get("ALERT_CHANNEL"),
    "webhook_url": dbutils.widgets.get("WEBHOOK_URL"),
}

# Generate pipeline run ID for tracking
PIPELINE_RUN_ID = str(uuid.uuid4())

if not CONFIG["kafka_bootstrap"]:
    raise ValueError("KAFKA_BOOTSTRAP must be provided")

drift_table_fqn = f"{CONFIG['catalog']}.{CONFIG['monitoring_schema']}.schema_drift_log"

In [ ]:
# Initialize monitoring table for schema drift tracking
create_schema_drift_table(spark, CONFIG['catalog'], CONFIG['monitoring_schema'])

# Initialize schema
ensure_schema_exists(CONFIG['catalog'], CONFIG['bronze_schema'])

In [ ]:
# Read from Kafka
kafka_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", CONFIG["kafka_bootstrap"])
    .option("subscribePattern", CONFIG["topic_pattern"])
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

# Parse Kafka metadata and extract table information from topic
bronze_events = (
    kafka_stream
    .select(
        col("topic"),
        col("partition"),
        col("offset"),
        to_timestamp(col("timestamp")).alias("kafka_timestamp"),
        col("key").cast("string").alias("message_key"),
        col("value").cast("string").alias("value"),
        current_timestamp().alias("ingested_at")
    )
    # Extract schema name from topic: cdc.{schema}.{table}
    .withColumn("source_schema", regexp_extract(col("topic"), r"^[^.]+\.([^.]+)\.[^.]+$", 1))
    # Extract table name from topic: cdc.{schema}.{table}
    .withColumn("table_name", regexp_extract(col("topic"), r"^[^.]+\.[^.]+\.([^.]+)$", 1))
    .filter(col("table_name") != "")
)

In [ ]:
# ============================================================================
# SCHEMA DRIFT VALIDATION FOR KAFKA TOPICS
# ============================================================================

# Get distinct topics being ingested
topics_df = bronze_events.select("topic", "table_name").distinct()
topics = [row.topic for row in topics_df.collect()]

print(f"Detected Kafka topics: {topics}")

# Validate schema for each topic
for topic in topics:
    table_name = topic.replace("cdc.public.", "")
    target_table = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_name}"
    
    # Get expected CDC payload schema
    cdc_contract_key = f"cdc.{table_name}"
    
    if cdc_contract_key in SCHEMA_CONTRACTS:
        expected_schema = SCHEMA_CONTRACTS[cdc_contract_key]
        
        # Read sample from existing Bronze table if it exists
        try:
            existing_df = spark.table(target_table)
            actual_schema = existing_df.schema
            
            print(f"\nValidating schema for {target_table}...")
            
            try:
                is_valid, drift_result = validate_schema_with_policy(
                    spark,
                    expected_schema=expected_schema,
                    actual_schema=actual_schema,
                    table_name=target_table,
                    drift_table=drift_table_fqn,
                    source_system="postgres_cdc",
                    pipeline_run_id=PIPELINE_RUN_ID,
                    policy=CONFIG['schema_policy'],
                    alert_channel=CONFIG['alert_channel'],
                    webhook_url=CONFIG['webhook_url'] if CONFIG['webhook_url'] else None
                )
                
                if drift_result['has_drift']:
                    print(f"  ⚠️ Schema drift detected: {drift_result['severity']}")
                else:
                    print(f"  ✅ Schema validation passed")
                    
            except SchemaDriftException as e:
                print(f"  🚨 BLOCKED: {e}")
                dbutils.notebook.exit(f"Schema drift blocked pipeline for {table_name}: {e}")
                
        except Exception as e:
            # Table doesn't exist yet - will be created
            print(f"  ℹ️ New table {target_table} - will be created with initial schema")
    else:
        print(f"  ⚠️ No schema contract found for {cdc_contract_key}")

In [ ]:
def write_bronze(batch_df, batch_id):
    """
    Write batch to Bronze tables with schema drift logging.
    """
    if not batch_df.take(1):
        return

    # Get distinct tables in this batch
    tables = [row.table_name for row in batch_df.select("table_name").distinct().collect()]

    for table_name in tables:
        target = f"{CONFIG['catalog']}.{CONFIG['bronze_schema']}.{table_name}"
        
        # Filter for this table
        table_df = batch_df.filter(col("table_name") == table_name)
        
        # Check schema before write
        batch_schema = table_df.schema
        
        # Write to Delta with schema evolution
        (
            table_df
            .write
            .format("delta")
            .mode("append")
            .option("mergeSchema", "true")
            .saveAsTable(target)
        )
        
        print(f"Wrote {table_df.count()} records to {target}")

In [ ]:
# Start streaming write
query = (
    bronze_events.writeStream
    .foreachBatch(write_bronze)
    .option("checkpointLocation", CONFIG["checkpoint_path"])
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .start()
)

# Wait for completion
query.awaitTermination()

print(f"✅ Bronze ingestion completed. Run ID: {PIPELINE_RUN_ID}")